# Headroom: Vorher / Nachher

Dieses Notebook zeigt Headroom so, wie man es im Video verständlich erklären kann:
- erst der rohe Kontext
- dann derselbe Kontext nach Headroom
- gleiche Ausgabeform
- klare Token-Ersparnis

## 1) Die drei Wege kurz

- `compress()` = direkt im Code
- `proxy` = Headroom sitzt als lokaler Zwischenserver davor
- `SDK / wrapper` = Headroom wird um den bestehenden Client gelegt

Für dieses Notebook nutzen wir nur `compress()`, weil man damit Vorher / Nachher am klarsten zeigen kann.

In [2]:
import json

import tiktoken
from headroom import compress

MODEL = "gpt-4o"

def token_count(messages, model=MODEL):
    try:
        enc = tiktoken.encoding_for_model(model)
    except Exception:
        enc = tiktoken.get_encoding("o200k_base")
    payload = json.dumps(messages, ensure_ascii=False, sort_keys=True)
    return len(enc.encode(payload))

def build_incident_context():
    incident_logs = "\n".join([
        f"2026-06-23 10:14:{i:02d} payment-service ERROR checkout_id=chk_{1000+i} provider=stripe status=502 retry={i%3} message=upstream timeout from payments-api"
        for i in range(120)
    ])

    support_tickets = json.dumps([
        {
            "ticket_id": f"CS-{2000 + i}",
            "customer_tier": "enterprise" if i % 4 == 0 else "pro",
            "channel": "chat" if i % 2 == 0 else "email",
            "issue_type": "payment_failed",
            "status": "waiting_for_retry",
            "country": ["DE", "AT", "CH", "NL"][i % 4],
            "message": "I tried to pay and got an error. Please check my invoice.",
        }
        for i in range(180)
    ], indent=2)

    root_cause_notes = "\n".join([
        "Incident summary: Stripe returned intermittent 502s on the payment authorization endpoint.",
        "Impact: checkout success rate dropped from 98.7% to 81.4% for 17 minutes.",
        "Timeline:",
        "- 10:12 UTC: first spike in failed payment attempts",
        "- 10:14 UTC: customer support started receiving duplicate complaints",
        "- 10:15 UTC: retries increased load on the upstream payments API",
        "- 10:19 UTC: fallback route engaged, incident stabilized",
        "Hypothesis: upstream network issue caused transient failures, not a code regression.",
        "Next step: correlate with provider status page and retry policy metrics.",
        "Open questions: was the failure regional, and did idempotency affect retries?",
    ])

    return [
        {"role": "system", "content": "You are helping investigate a payment incident. Keep only high-signal facts."},
        {"role": "user", "content": "Please analyze the incident material and summarize what matters for the on-call response."},
        {"role": "assistant", "content": "I will inspect the logs, support tickets, and root-cause notes."},
        {"role": "tool", "tool_call_id": "logs", "content": incident_logs},
        {"role": "tool", "tool_call_id": "tickets", "content": support_tickets},
        {"role": "tool", "tool_call_id": "notes", "content": root_cause_notes},
    ]

def print_context(title, messages):
    print(title)
    print(f"TOKENS: {token_count(messages)}")
    print("CONTEXT")
    for msg in messages:
        role = msg.get("role", "unknown")
        content = str(msg.get("content", "")).replace("\n", " ")
        preview = content[:160]
        print(f"- {role}: {preview}")


## 2) Vor Headroom

Hier zeigen wir den Rohkontext. Genau so kommt viel Text, viele Wiederholungen und viel Noise in den Prompt.

In [3]:
messages = build_incident_context()
print_context("RAW", messages)


RAW
TOKENS: 20560
CONTEXT
- system: You are helping investigate a payment incident. Keep only high-signal facts.
- user: Please analyze the incident material and summarize what matters for the on-call response.
- assistant: I will inspect the logs, support tickets, and root-cause notes.
- tool: 2026-06-23 10:14:00 payment-service ERROR checkout_id=chk_1000 provider=stripe status=502 retry=0 message=upstream timeout from payments-api 2026-06-23 10:14:01
- tool: [   {     "ticket_id": "CS-2000",     "customer_tier": "enterprise",     "channel": "chat",     "issue_type": "payment_failed",     "status": "waiting_for_retry
- tool: Incident summary: Stripe returned intermittent 502s on the payment authorization endpoint. Impact: checkout success rate dropped from 98.7% to 81.4% for 17 minu


## 3) Headroom einsetzen

Jetzt komprimieren wir denselben Kontext. Für die Demo nehmen wir eine eher aggressive Einstellung, damit der Effekt klar sichtbar wird.

In [4]:
messages = build_incident_context()
raw_tokens = token_count(messages)

result = compress(
    messages,
    model=MODEL,
    compress_user_messages=True,
    protect_recent=0,
    target_ratio=0.3,
    min_tokens_to_compress=0,
)

compressed_tokens = token_count(result.messages)
tokens_saved = raw_tokens - compressed_tokens
saved_pct = (tokens_saved / raw_tokens * 100) if raw_tokens else 0

print_context("COMPRESSED", result.messages)
print(f"SAVED TOKENS: {tokens_saved}")
print(f"SAVED PCT: {saved_pct:.1f}%")
print("TRANSFORMS")
print(", ".join(result.transforms_applied) if result.transforms_applied else "none")


COMPRESSED
TOKENS: 6744
CONTEXT
- system: You are helping investigate a payment incident. Keep only high-signal facts.
- user: Please analyze the incident material and summarize what matters for the on-call response.
- assistant: I will inspect the logs, support tickets, and root-cause notes.
- tool: 2026-06-23 10:14:00 payment-service ERROR checkout_id=chk_1000 provider=stripe status=502 retry=0 message=upstream timeout from payments-api 2026-06-23 10:14:01
- tool: "[180]{channel:string,country:string,customer_tier:string,issue_type:string,message:string,status:string,ticket_id:string}\nchat,DE,enterprise,payment_failed,I 
- tool: Incident summary: Stripe returned intermittent 502s on the payment authorization endpoint. Impact: checkout success rate dropped from 98.7% to 81.4% for 17 minu
SAVED TOKENS: 13816
SAVED PCT: 67.2%
TRANSFORMS
router:log:0.14, router:mixed:0.41


## 4) Live OpenAI: Vorher vs. Nachher

Wir schicken **denselben Prompt zweimal** an OpenAI – einmal mit dem Rohkontext, einmal mit dem Headroom-komprimierten Kontext.
Beide Antworten kommen von OpenAI. Die Zahlen (prompt_tokens, latency) kommen aus der API-Response, nicht aus tiktoken.
So sieht man: gleiche Antwortqualität, deutlich weniger Token und weniger Latenz.


In [3]:
import os
import time
from dotenv import load_dotenv
from openai import OpenAI
from headroom import compress

# Rebuild context and result if this cell is run before the cells above
if "build_incident_context" not in dir() or "result" not in dir():
    import json as _json, tiktoken as _tiktoken
    _MODEL = "gpt-4o"

    def build_incident_context():
        incident_logs = "\n".join([
            f"2026-06-23 10:14:{i:02d} payment-service ERROR checkout_id=chk_{1000+i} provider=stripe status=502 retry={i%3} message=upstream timeout from payments-api"
            for i in range(120)
        ])
        support_tickets = _json.dumps([
            {"ticket_id": f"CS-{2000+i}", "customer_tier": "enterprise" if i%4==0 else "pro",
             "channel": "chat" if i%2==0 else "email", "issue_type": "payment_failed",
             "status": "waiting_for_retry", "country": ["DE","AT","CH","NL"][i%4],
             "message": "I tried to pay and got an error. Please check my invoice."}
            for i in range(180)
        ], indent=2)
        root_cause_notes = "\n".join([
            "Incident summary: Stripe returned intermittent 502s on the payment authorization endpoint.",
            "Impact: checkout success rate dropped from 98.7% to 81.4% for 17 minutes.",
            "Timeline:",
            "- 10:12 UTC: first spike in failed payment attempts",
            "- 10:14 UTC: customer support started receiving duplicate complaints",
            "- 10:15 UTC: retries increased load on the upstream payments API",
            "- 10:19 UTC: fallback route engaged, incident stabilized",
            "Hypothesis: upstream network issue caused transient failures, not a code regression.",
            "Next step: correlate with provider status page and retry policy metrics.",
            "Open questions: was the failure regional, and did idempotency affect retries?",
        ])
        return [
            {"role": "system", "content": "You are helping investigate a payment incident. Keep only high-signal facts."},
            {"role": "user", "content": "Please analyze the incident material and summarize what matters for the on-call response."},
            {"role": "assistant", "content": "I will inspect the logs, support tickets, and root-cause notes."},
            {"role": "tool", "tool_call_id": "logs", "content": incident_logs},
            {"role": "tool", "tool_call_id": "tickets", "content": support_tickets},
            {"role": "tool", "tool_call_id": "notes", "content": root_cause_notes},
        ]

    _msgs = build_incident_context()
    result = compress(
        _msgs, model=_MODEL,
        compress_user_messages=True, protect_recent=0,
        target_ratio=0.3, min_tokens_to_compress=0,
    )

load_dotenv(dotenv_path=".env", override=False)
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Missing OPENAI_API_KEY in .env")

model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
client = OpenAI(api_key=api_key)

SYSTEM = "You are a concise on-call assistant. Answer only from the provided context."
QUESTION = "Give me a 2-bullet summary: what failed, and what was the customer impact?"


def flatten_context(messages):
    """Merge all non-system messages into one text block for a fair comparison."""
    parts = []
    for msg in messages:
        role = msg.get("role", "unknown")
        content = str(msg.get("content", ""))
        if role != "system":
            parts.append(f"[{role.upper()}]\n{content}")
    return "\n\n".join(parts)


def call_openai(label, context_text):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"{context_text}\n\n{QUESTION}"},
    ]
    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=msgs,
        max_tokens=150,
    )
    latency_ms = (time.perf_counter() - t0) * 1000
    u = response.usage
    return {
        "label": label,
        "latency_ms": round(latency_ms),
        "prompt_tokens": u.prompt_tokens,
        "completion_tokens": u.completion_tokens,
        "total_tokens": u.total_tokens,
        "reply": response.choices[0].message.content,
    }


raw_text = flatten_context(build_incident_context())
compressed_text = flatten_context(result.messages)

print(f"Calling OpenAI [{model}] with RAW context  ({result.tokens_before:,} tokens)...")
before = call_openai("RAW", raw_text)

print(f"Calling OpenAI [{model}] with HEADROOM context ({result.tokens_after:,} tokens)...")
after = call_openai("HEADROOM", compressed_text)

# --- comparison table ---
pt_saved = before["prompt_tokens"] - after["prompt_tokens"]
pt_pct = pt_saved / before["prompt_tokens"] * 100
lat_saved = before["latency_ms"] - after["latency_ms"]

print()
print(f"{'Metric':<22} {'RAW':>10} {'HEADROOM':>10} {'Saved':>10} {'%':>7}")
print("-" * 63)
print(f"{'prompt_tokens':<22} {before['prompt_tokens']:>10,} {after['prompt_tokens']:>10,} {pt_saved:>10,} {pt_pct:>6.1f}%")
print(f"{'completion_tokens':<22} {before['completion_tokens']:>10,} {after['completion_tokens']:>10,}")
print(f"{'total_tokens':<22} {before['total_tokens']:>10,} {after['total_tokens']:>10,}")
print(f"{'latency_ms':<22} {before['latency_ms']:>10,} {after['latency_ms']:>10,} {lat_saved:>10,}")

print()
print("=== RAW reply ===")
print(before["reply"])
print()
print("=== HEADROOM reply ===")
print(after["reply"])


Calling OpenAI [gpt-4o-mini] with RAW context  (18,518 tokens)...
Calling OpenAI [gpt-4o-mini] with HEADROOM context (6,502 tokens)...

Metric                        RAW   HEADROOM      Saved       %
---------------------------------------------------------------
prompt_tokens              18,539      6,522     12,017   64.8%
completion_tokens              60         52
total_tokens               18,599      6,574
latency_ms                  6,441      1,584      4,857

=== RAW reply ===
- Payment authorization through Stripe intermittently returned 502 errors, leading to a significant drop in checkout success rate from 98.7% to 81.4%.
- Customers experienced payment failures, with support tickets indicating widespread issues across multiple countries, prompting retries that intensified the load on the upstream API.

=== HEADROOM reply ===
- The payment authorization endpoint from Stripe encountered intermittent 502 errors, leading to a spike in failed transactions.
- The checkout succ

## 5) Was steuert die Trim-Entscheidung?

Die wichtigsten Stellschrauben sind:
- `compress_user_messages`
- `compress_system_messages`
- `protect_recent`
- `target_ratio`
- `min_tokens_to_compress`
- `savings_profile`

Wenn du es feiner brauchst, kommen Hooks dazu. Für dieses Notebook reicht aber die Vorher/Nachher-Demo völlig.

## 6) Merksatz

Headroom ist nicht das Modell.
Headroom sitzt davor und macht aus großem Kontext einen kleineren, günstigeren und fokussierteren Kontext.